In [ ]:
import pandas as pd
import re
import numpy as np

df = pd.read_csv('./facteus_10k_user_panel.csv')

In [ ]:
def identify_merchant(merchant_string):
    word = 'OTHER'
    if not isinstance(merchant_string, str):
        return word
    if re.search(r'.*MCDONALD.*', merchant_string, re.IGNORECASE):
        word = 'MCDONALDS'
    elif re.search(r'.*(AM[A]?Z[O]?N|PRIME|KINDLE).*', merchant_string, re.IGNORECASE):
        word = 'AMAZON'
    elif re.search(r'.*(APPLE|ITUNES).*', merchant_string, re.IGNORECASE):
        word = 'APPLE'
    return word

df['identified_merchant'] = df['merchant_string_example'].apply(identify_merchant)
df

In [ ]:
import pandas as pd
import numpy as np

def create_ticker_to_identified_merchants_map(dataframe):
    ticker_to_merchants = {}
    no_ticker_set = set()

    for index, row in dataframe.iterrows():
        merchant_ticker = row['merchant_ticker']
        identified_merchant = row['merchant']

        if pd.notna(merchant_ticker) and pd.notna(identified_merchant) and isinstance(identified_merchant, str):
            if merchant_ticker not in ticker_to_merchants:
                ticker_to_merchants[merchant_ticker] = set()
            ticker_to_merchants[merchant_ticker].add(identified_merchant)
        else:
          no_ticker_list.append(identified_merchant)

    final_map = {ticker: list(merchants) for ticker, merchants in ticker_to_merchants.items()}
    return final_map, no_ticker_list

ticker_to_identified_merchants_map, no_ticker_list = create_ticker_to_identified_merchants_map(df)


In [ ]:
import pickle

with open('ticker_to_identified_merchants_map.pkl', 'wb') as f:
    pickle.dump(ticker_to_identified_merchants_map, f)

In [ ]:
import pickle

with open('ticker_to_identified_merchants_map.pkl', 'rb') as f:
    ticker_to_identified_merchants_map = pickle.load(f)
ticker_to_identified_merchants_map


In [ ]:
from pytrends.request import TrendReq
import pandas as pd
import time
import random
import math

def safe_build_payload(pytrends, kw, timeframe, geo, max_retries=5):
    for attempt in range(1, max_retries + 1):
        try:
            pytrends.build_payload([kw], timeframe=timeframe, geo=geo)
            return True

        except Exception as e:
            msg = str(e).lower()
            print(f"[Attempt {attempt}] Failed payload for '{kw}': {e}")

            if "429" in msg or "too many requests" in msg:
                sleep_time = random.uniform(40, 60)
                print(f"429 encountered — sleeping {sleep_time:.2f}s...")
            else:
                sleep_time = random.uniform(5, 10)
                print(f"Sleeping {sleep_time:.2f}s...")

            time.sleep(sleep_time)

    print(f"Giving up on keyword '{kw}'.")
    return False

def fetch_aggregated_trends_in_batches(
    ticker_terms_map,
    timeframe="2020-01-06 2020-12-31",
    geo="US",
    base_csv_name="weekly_aggregated_batch",
    save_dir=".",
    start_batch=0
):

    pytrends = TrendReq(hl='en-US', tz=360)

    merchants = list(ticker_terms_map.items())
    batch_size = 15
    num_batches = math.ceil(len(merchants) / batch_size)

    print(f"Total merchants: {len(merchants)}")
    print(f"Processing in {num_batches} batches of {batch_size} each.\n")

    for batch_idx in range(num_batches):
        if batch_idx < start_batch:
            continue
        start = batch_idx * batch_size
        end = start + batch_size
        batch_merchants = merchants[start:end]

        print(f"\n==============================")
        print(f"   Processing batch {batch_idx+1}/{num_batches}")
        print(f"   Merchants {start+1} to {min(end, len(merchants))}")
        print("==============================\n")

        batch_results = []

        for ticker, terms in batch_merchants:

            kws = [ticker] + terms
            dfs = []

            print(f"\n--- {ticker}: {kws} ---")

            for kw in kws:

                if not safe_build_payload(pytrends, kw, timeframe, geo):
                    continue

                try:
                    df = pytrends.interest_over_time()
                except Exception as e:
                    print(f"Error retrieving '{kw}': {e}")
                    continue

                if df.empty:
                    print(f"No data for '{kw}'. Skipping.")
                    continue

                df = df[[kw]]
                dfs.append(df.rename(columns={kw: f"{kw}"}))

            if not dfs:
                print(f"No usable data for ticker {ticker}.")
                continue

            combined = pd.concat(dfs, axis=1)
            combined["value"] = combined.sum(axis=1)

            out = combined[["value"]].copy()
            out["ticker"] = ticker
            out = out.reset_index()

            batch_results.append(out)
            # normal sleep
            pause = random.uniform(5, 10)
            print(f"Sleeping {pause:.2f}s after request...")
            time.sleep(pause)

        if batch_results:
            batch_df = pd.concat(batch_results, ignore_index=True)
            batch_df = batch_df[["ticker", "date", "value"]]

            filename = f"{save_dir}/{base_csv_name}_batch_{batch_idx+1}.csv"
            batch_df.to_csv(filename, index=False)


            print(f"\nSaved batch {batch_idx+1} to: {filename}\n")
        else:
            print(f"No data in batch {batch_idx+1}, skipping save.")

fetch_aggregated_trends_in_batches(
    ticker_to_identified_merchants_map,
    base_csv_name="weekly_all_aggregated_trends_2020",
    start_batch=5
)


In [ ]:
import pandas as pd
import glob
import os

csv_folder = "."
csv_files = glob.glob(os.path.join(csv_folder, "weekly*.csv"))

dfs = []

for file in csv_files:
    df = pd.read_csv(file)
    dfs.append(df)

combined_df = pd.concat(dfs, ignore_index=True)
combined_file = os.path.join(csv_folder, "all_trends_combined.csv")
combined_df.to_csv(combined_file, index=False)

print(f"Combined {len(csv_files)} CSVs into {combined_file}")
print(f"Total rows: {len(combined_df)}")


In [ ]:
import pandas as pd
from datetime import timedelta
import sys

input_file = 'all_trends_combined.csv'
output_file = 'all_trends_daily.csv'

df = pd.read_csv(input_file)
df['date'] = pd.to_datetime(df['date'])

daily_rows = []

for idx, row in df.iterrows():
    ticker = row['ticker']
    start_date = row['date']
    weekly_value = row['value']
    daily_value = weekly_value / 7
    
    for i in range(7):
        current_date = start_date + timedelta(days=i)
        daily_rows.append({
            'ticker': ticker,
            'date': current_date,
            'value': daily_value
        })

daily_df = pd.DataFrame(daily_rows)
daily_df = daily_df.sort_values(['ticker', 'date']).reset_index(drop=True)

start_filter = pd.to_datetime('2020-01-06')
end_filter = pd.to_datetime('2020-12-31')
daily_df = daily_df[(daily_df['date'] >= start_filter) & (daily_df['date'] <= end_filter)]

daily_df.to_csv(output_file, index=False)
print(f"Converted {len(df)} weekly rows to {len(daily_df)} daily rows")
print(f"Output saved to {output_file}")